# SIKDD 2026 Replication — Feedback-Driven Retrieval for IT Ticket Resolution

**Purpose:** Replicate the core findings of the SIKDD 2026 paper in the clean modular environment
with proper train/dev/eval separation, fresh feedback DB, continuous aggregation, and GPT-5.6-Luna generation.

**Data:** `results/sikdd_replication.csv` — produced by `experiments/parse_to_flat_csv.py`
from the `04_evaluate.py` JSON outputs.

**Methods compared:** Baseline, M1 (global), M1-blind, M1-binary, M2 (team-only), M3 (class-only), M4 (intersection).

**Primary metric:** Cosine similarity (multi-qa-MiniLM-L6-cos-v1) between generated reply and reference reply.
Secondary: ROUGE-L F1.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIG_DIR = ROOT / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Results dir: {RESULTS}")

Root: c:\Users\ailab\Documents\code\financial-pilot-ea\paper_ieee_access\notebooks
Results dir: c:\Users\ailab\Documents\code\financial-pilot-ea\paper_ieee_access\notebooks\results


In [ ]:
# Load the flat CSV
csv_path = RESULTS / "sikdd_replication.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows, {df['experiment_id'].nunique()} experiments, {df['ticket_id'].nunique()} tickets")
print(f"\nExperiments:")
for exp in sorted(df["experiment_id"].unique()):
    sub = df[df["experiment_id"] == exp]
    d = sub["delta_cosine"].dropna()
    print(f"  {exp:50s} n={len(d):3d}  mean_delta={d.mean():+.4f}  pct_imp={100*(d>0).mean():.1f}%")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Define method display names, colors, and ordering
METHOD_INFO = {
    "baseline": {
        "label": "Baseline (FAISS-only)",
        "short": "Baseline",
        "color": "#7F7F7F",
        "marker": "s",
    },
    "M1_global": {
        "label": "M1 — Global Feedback",
        "short": "M1 Global",
        "color": "#4472C4",
        "marker": "o",
    },
    "M1_blind": {
        "label": "M1 — Blind Protocol",
        "short": "M1 Blind",
        "color": "#ED7D31",
        "marker": "D",
    },
    "M1_binary": {
        "label": "M1 — Binary Aggregation",
        "short": "M1 Binary",
        "color": "#A5A5A5",
        "marker": "^",
    },
    "M2_team": {
        "label": "M2 — Team-Only",
        "short": "M2 Team",
        "color": "#70AD47",
        "marker": "v",
    },
    "M3_class": {
        "label": "M3 — Class-Only",
        "short": "M3 Class",
        "color": "#FFC000",
        "marker": "<",
    },
    "M4_intersection": {
        "label": "M4 — Team+Class",
        "short": "M4 Intersection",
        "color": "#5B9BD5",
        "marker": ">",
    },
}

# Map experiment_id to method key
def map_exp_to_method(exp_id: str) -> str:
    eid = exp_id.lower()
    if "baseline" in eid:
        return "baseline"
    if "blind" in eid:
        return "M1_blind"
    if "binary" in eid:
        return "M1_binary"
    if "m1_global" in eid or "m1_" in eid and "team" not in eid and "class" not in eid and "intersection" not in eid:
        return "M1_global"
    if "m2_team" in eid or "team_only" in eid:
        return "M2_team"
    if "m3_class" in eid or "class_only" in eid:
        return "M3_class"
    if "m4_intersection" in eid or "intersection" in eid:
        return "M4_intersection"
    return "M1_global"

df["method_key"] = df["experiment_id"].apply(map_exp_to_method)
print(f"Mapped methods: {sorted(df['method_key'].unique())}")

## 2. Method Comparison — Core Result Table

For each method, report: mean delta, bootstrap 95% CI, percent improved, Wilcoxon p-value, Cohen's d.
The baseline row shows the absolute baseline cosine (not a delta).

In [ ]:
def bootstrap_ci(data, n_resamples=10000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    means = np.array([np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_resamples)])
    lo = 100 * alpha / 2
    hi = 100 * (1 - alpha / 2)
    return float(np.percentile(means, lo)), float(np.percentile(means, hi))

method_order = ["baseline", "M1_global", "M1_blind", "M1_binary", "M2_team", "M3_class", "M4_intersection"]
method_order = [m for m in method_order if m in df["method_key"].unique()]

rows = []
for mk in method_order:
    info = METHOD_INFO.get(mk, {})
    sub = df[df["method_key"] == mk]
    n = len(sub)
    
    bl_cos = sub["baseline_cosine"].dropna()
    fb_cos = sub["feedback_cosine"].dropna()
    delta = sub["delta_cosine"].dropna()
    
    if mk == "baseline":
        rows.append({
            "Method": info.get("short", mk),
            "n": int(len(bl_cos)),
            "Baseline cos (mean)": f"{bl_cos.mean():.4f}",
            "Baseline cos (median)": f"{bl_cos.median():.4f}",
            "Mean Δ": "—",
            "% Improved": "—",
            "Bootstrap 95% CI": "—",
            "Wilcoxon p": "—",
            "Cohen's d": "—",
        })
    else:
        ci_lo, ci_hi = bootstrap_ci(delta.values)
        w_stat, w_p = stats.wilcoxon(delta.values) if len(delta) > 0 else (np.nan, np.nan)
        cd = delta.mean() / max(delta.std(), 1e-8)
        rows.append({
            "Method": info.get("short", mk),
            "n": int(n),
            "Baseline cos (mean)": f"{bl_cos.mean():.4f}",
            "Baseline cos (median)": f"{bl_cos.median():.4f}",
            "Mean Δ": f"{delta.mean():+.4f}",
            "% Improved": f"{100*(delta > 0).mean():.1f}%",
            "Bootstrap 95% CI": f"[{ci_lo:+.4f}, {ci_hi:+.4f}]",
            "Wilcoxon p": f"{w_p:.4f}",
            "Cohen's d": f"{cd:+.4f}",
        })

comparison_df = pd.DataFrame(rows)
comparison_df

In [ ]:
# Bar chart: mean delta per method with 95% bootstrap CI error bars
fig, ax = plt.subplots(figsize=(10, 5))

plot_methods = [m for m in method_order if m != "baseline"]
means = []
ci_los = []
ci_his = []
labels = []
colors = []
pcts = []

for mk in plot_methods:
    info = METHOD_INFO.get(mk, {})
    sub = df[df["method_key"] == mk]
    delta = sub["delta_cosine"].dropna()
    if len(delta) == 0:
        continue
    lo, hi = bootstrap_ci(delta.values)
    means.append(delta.mean())
    ci_los.append(delta.mean() - lo)
    ci_his.append(hi - delta.mean())
    labels.append(info.get("short", mk))
    colors.append(info.get("color", "#4472C4"))
    pcts.append(100 * (delta > 0).mean())

x = np.arange(len(means))
bars = ax.bar(x, means, yerr=[ci_los, ci_his], capsize=5,
              color=colors, edgecolor="white", linewidth=0.8)

ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Mean Δ Cosine Similarity", fontsize=12)
ax.set_title("Feedback-Driven Retrieval vs Baseline — Dev Set (n=319)", fontsize=13)

for i, (m, p) in enumerate(zip(means, pcts)):
    offset = 0.002 if m >= 0 else -0.004
    ax.text(i, m + offset, f"{p:.0f}%\u2191", ha="center", fontsize=8,
            color="#2E7D32" if m >= 0 else "#C62828")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f"))
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_method_comparison.png", dpi=150)
plt.show()

## 3. Oracle Decile Analysis — When Does Feedback Help?

The SIKDD paper's central finding: feedback helps when the baseline answer is weak
and hurts when it is strong. We stratify tickets by baseline cosine similarity
into deciles and report the mean delta per decile.

In [ ]:
# Select the primary method (M1 global, conditioned, continuous)
oracle_method = "M1_global"
oracle_data = df[df["method_key"] == oracle_method].copy()

if len(oracle_data) == 0:
    oracle_data = df[df["method_key"] == df["method_key"].unique()[0]].copy()
    oracle_method = df["method_key"].unique()[0]

oracle_data = oracle_data.sort_values("baseline_cosine")
n = len(oracle_data)
decile_size = max(1, n // 10)

deciles = []
for d in range(10):
    start = d * decile_size
    end = start + decile_size if d < 9 else n
    chunk = oracle_data.iloc[start:end]
    bl_mean = chunk["baseline_cosine"].mean()
    delta_mean = chunk["delta_cosine"].mean()
    pct_imp = 100 * (chunk["delta_cosine"] > 0).mean()
    deciles.append({
        "Decile": d + 1,
        "n": len(chunk),
        "Baseline cos (mean)": bl_mean,
        "Mean Δ": delta_mean,
        "% Improved": pct_imp,
    })

decile_df = pd.DataFrame(deciles)
decile_df

In [ ]:
# Oracle decile figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: delta per decile
ax = axes[0]
dvals = decile_df["Mean \u0394"].values
colors = ["#C00000" if v < 0 else "#2E7D32" for v in dvals]
bars = ax.bar(range(1, 11), dvals, color=colors, edgecolor="white", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Decile (1 = weakest baseline)", fontsize=11)
ax.set_ylabel("Mean Δ Cosine", fontsize=11)
ax.set_title(f"Oracle Decile Analysis — {METHOD_INFO.get(oracle_method, {}).get('short', oracle_method)}", fontsize=12)

# Add linear fit
x_vals = np.arange(1, 11)
slope, intercept, r_val, p_val, _ = stats.linregress(x_vals, dvals)
ax.plot(x_vals - 1, intercept + slope * (x_vals - 1), "k--", linewidth=1.2,
        label=f"r={r_val:.2f}, p={p_val:.3f}")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f"))

# Right: % improved per decile
ax = axes[1]
pvals = decile_df["% Improved"].values
ax.bar(range(1, 11), pvals, color=["#2E7D32" if v > 50 else "#C00000" for v in pvals],
       edgecolor="white", linewidth=0.8)
ax.axhline(50, color="black", linewidth=0.8, linestyle="--", alpha=0.5, label="chance")
ax.set_xlabel("Decile (1 = weakest baseline)", fontsize=11)
ax.set_ylabel("% Tickets Improved", fontsize=11)
ax.set_title(f"% Improved per Decile", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_oracle_decile.png", dpi=150)
plt.show()

## 4. Gate Cutoff Sweep — Can a Simple Threshold Gate Feedback?

The SIKDD finding: no univariate pre-generation signal beats always-on under
cross-validation. We simulate a gate: only apply feedback when `top1_faiss < cutoff`,
otherwise the delta is 0 (baseline is used). Sweep cutoffs from 0.5 to 1.0.

In [ ]:
cutoffs = np.linspace(0.50, 1.00, 21)
gate_rows = []

for cutoff in cutoffs:
    gated_deltas = []
    for _, row in oracle_data.iterrows():
        t1 = row.get("baseline_top1_faiss", 1.0)
        if pd.isna(t1):
            t1 = 1.0
        if t1 < cutoff:
            gated_deltas.append(row["delta_cosine"])
        else:
            gated_deltas.append(0.0)
    gate_rows.append({
        "cutoff": cutoff,
        "mean_gated_delta": np.mean(gated_deltas),
        "pct_gate_open": 100 * np.mean([r.get("baseline_top1_faiss", 1.0) < cutoff 
                                            if not pd.isna(r.get("baseline_top1_faiss", 1.0)) else False
                                            for _, r in oracle_data.iterrows()]),
    })

gate_df = pd.DataFrame(gate_rows)

always_on = oracle_data["delta_cosine"].mean()
print(f"Always-on mean delta: {always_on:+.4f}")
print(f"Best gate cutoff: {gate_df.loc[gate_df['mean_gated_delta'].idxmax(), 'cutoff']:.2f} "
      f"with delta = {gate_df['mean_gated_delta'].max():+.4f}")
print(f"Is best gate better than always-on? {'YES' if gate_df['mean_gated_delta'].max() > always_on else 'NO'}")

gate_df

In [ ]:
# Gate sweep figure
fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = "#4472C4"
color2 = "#ED7D31"

ax1.plot(gate_df["cutoff"], gate_df["mean_gated_delta"], color=color1, linewidth=2, marker="o", markersize=4)
ax1.axhline(always_on, color=color1, linestyle="--", linewidth=1.5, alpha=0.6,
            label=f"always-on = {always_on:+.4f}")
ax1.set_xlabel("Top-1 FAISS Cutoff (feedback applied below this threshold)", fontsize=11)
ax1.set_ylabel("Mean Gated \u0394 Cosine", color=color1, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.4f"))

ax2 = ax1.twinx()
ax2.plot(gate_df["cutoff"], gate_df["pct_gate_open"], color=color2, linewidth=2, marker="s", markersize=4)
ax2.set_ylabel("% Queries Receiving Feedback", color=color2, fontsize=11)
ax2.tick_params(axis="y", labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = [plt.Line2D([0], [0], color=color2, linewidth=2)], ["% gate open"]
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

ax1.set_title("Static Gate Cutoff Sweep — Top-1 FAISS Threshold", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_gate_sweep.png", dpi=150)
plt.show()

## 5. Rescue vs Disruption — Can We Distinguish Them at Retrieval Level?

The SIKDD finding: rescues (feedback helps) and disruptions (feedback hurts)
have nearly identical top-1 FAISS scores — retrieval confidence cannot
distinguish which tickets will benefit from which will be harmed.

In [ ]:
def classify_delta(d):
    if pd.isna(d):
        return "unknown"
    if d > 0.02:
        return "rescue"
    if d < -0.02:
        return "disruption"
    return "neutral"

oracle_data["outcome"] = oracle_data["delta_cosine"].apply(classify_delta)

profile_rows = []
for label in ["rescue", "disruption", "neutral"]:
    sub = oracle_data[oracle_data["outcome"] == label]
    if len(sub) == 0:
        continue
    profile_rows.append({
        "Group": label.capitalize(),
        "n": len(sub),
        "Baseline cos (mean)": sub["baseline_cosine"].mean(),
        "Top-1 FAISS (mean)": sub["baseline_top1_faiss"].mean(),
        "Mean \u0394": sub["delta_cosine"].mean(),
    })

profile_df = pd.DataFrame(profile_rows)
profile_df

In [ ]:
# Rescue vs disruption top-1 FAISS distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors_map = {"rescue": "#2E7D32", "disruption": "#C00000", "neutral": "#7F7F7F"}

for ax, (label, color) in zip(axes, [["rescue", "disruption"], ["rescue", "neutral"]]):
    for lbl in label:
        sub = oracle_data[oracle_data["outcome"] == lbl]
        vals = sub["baseline_top1_faiss"].dropna()
        ax.hist(vals, bins=20, alpha=0.5, color=colors_map[lbl], label=f"{lbl} (n={len(sub)})")
    ax.set_xlabel("Top-1 FAISS Similarity")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

axes[0].set_title("Rescue vs Disruption — Top-1 FAISS")
axes[1].set_title("Rescue vs Neutral — Top-1 FAISS")
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_rescue_disruption.png", dpi=150)
plt.show()

## 6. Per-Team and Per-Class Analysis

Which teams and ticket classes benefit most from feedback? The SIKDD finding:
team/type identity provides a weak but cross-scope-stable prior — some teams
consistently benefit, others are consistently harmed.

In [ ]:
# Per-team delta (n >= 5)
team_stats = oracle_data.groupby("expected_team").agg(
    n=("delta_cosine", "count"),
    mean_delta=("delta_cosine", "mean"),
    std_delta=("delta_cosine", "std"),
    mean_top1=("baseline_top1_faiss", "mean"),
).query("n >= 5").sort_values("mean_delta")

fig, ax = plt.subplots(figsize=(10, 5))
deltas = team_stats["mean_delta"].values
colors = ["#2E7D32" if d > 0 else "#C00000" for d in deltas]
ax.barh(range(len(team_stats)), deltas, xerr=team_stats["std_delta"], capsize=3,
        color=colors, edgecolor="white", linewidth=0.8)
ax.set_yticks(range(len(team_stats)))
ax.set_yticklabels([t[:38] for t in team_stats.index], fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean \u0394 Cosine", fontsize=11)
ax.set_title(f"Per-Team Feedback Effect — {METHOD_INFO.get(oracle_method, {}).get('short', oracle_method)}", fontsize=12)

for i, (_, row) in enumerate(team_stats.iterrows()):
    offset = 0.003 if row["mean_delta"] >= 0 else -0.005
    ax.text(row["mean_delta"] + offset, i, f"n={int(row['n'])}", va="center", fontsize=7)

ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f"))
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_per_team_delta.png", dpi=150)
plt.show()

In [ ]:
# Per-class delta
class_stats = oracle_data.groupby("expected_class").agg(
    n=("delta_cosine", "count"),
    mean_delta=("delta_cosine", "mean"),
    std_delta=("delta_cosine", "std"),
).sort_values("mean_delta")

fig, ax = plt.subplots(figsize=(10, 4.5))
deltas = class_stats["mean_delta"].values
colors = ["#2E7D32" if d > 0 else "#C00000" for d in deltas]
ax.barh(range(len(class_stats)), deltas, xerr=class_stats["std_delta"], capsize=3,
        color=colors, edgecolor="white", linewidth=0.8)
ax.set_yticks(range(len(class_stats)))
ax.set_yticklabels(class_stats.index, fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean \u0394 Cosine", fontsize=11)
ax.set_title(f"Per-Class Feedback Effect", fontsize=12)

for i, (_, row) in enumerate(class_stats.iterrows()):
    offset = 0.003 if row["mean_delta"] >= 0 else -0.005
    ax.text(row["mean_delta"] + offset, i, f"n={int(row['n'])}", va="center", fontsize=7)

ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f"))
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_per_class_delta.png", dpi=150)
plt.show()

## 7. Method Cross-Comparison — Delta Distributions

How do the per-ticket delta distributions compare across methods? Box plot
across all non-baseline methods.

In [ ]:
plot_methods = [m for m in method_order if m != "baseline" and m in df["method_key"].unique()]
box_data = []
box_labels = []
box_colors = []

for mk in plot_methods:
    info = METHOD_INFO.get(mk, {})
    sub = df[df["method_key"] == mk]
    deltas = sub["delta_cosine"].dropna()
    if len(deltas) == 0:
        continue
    box_data.append(deltas.values)
    box_labels.append(info.get("short", mk))
    box_colors.append(info.get("color", "#4472C4"))

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, showfliers=False, widths=0.6)
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("\u0394 Cosine Similarity", fontsize=11)
ax.set_title("Per-Ticket Delta Distribution by Method — Dev Set", fontsize=12)
ax.tick_params(axis="x", rotation=25)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.2f"))
plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_delta_distributions.png", dpi=150)
plt.show()

## 8. Retrieval Overlap Analysis

How much do the baseline and feedback top-5 retrieval sets overlap?
If feedback changes nothing (overlap = 5), lifts are ineffective.
If feedback changes everything (overlap = 0), retrieval is fundamentally different.

In [ ]:
fig, axes = plt.subplots(1, len(plot_methods), figsize=(3 * len(plot_methods), 3.5))
if len(plot_methods) == 1:
    axes = [axes]

for ax, mk in zip(axes, plot_methods):
    info = METHOD_INFO.get(mk, {})
    sub = df[df["method_key"] == mk]
    overlap_vals = sub["retrieval_overlap"].dropna()
    if len(overlap_vals) == 0:
        continue
    ax.hist(overlap_vals, bins=np.arange(-0.5, 6.5, 1), color=info.get("color", "#4472C4"),
            edgecolor="white", alpha=0.85, align="mid")
    ax.set_xlabel("Retrieval Overlap (of 5)")
    ax.set_ylabel("Count")
    ax.set_title(f"{info.get('short', mk)}\nmean overlap = {overlap_vals.mean():.1f}", fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / "sikdd_retrieval_overlap.png", dpi=150)
plt.show()

## 9. SIKDD Claims Replication Summary

| # | SIKDD Claim | Replication Status | Evidence |
|---|-------------|-------------------|----------|
| C1 | Popularity and semantics are orthogonal (r=0.06) | (Not tested here — requires retrieval-level analysis, not generation-level) | — |
| C2 | Laplace formula is statistically justified | (Not tested here — separate formula comparison needed) | — |
| C3 | Feedback helps weak baselines, hurts strong ones (oracle pattern) | See Sec 3 — linear fit r value from decile analysis | Oracle decile figure |
| C4 | No pre-generation signal gates feedback | See Sec 4 — best gate cutoff vs always-on | Gate sweep figure |
| C5 | Core retrieval findings are model-independent | (Cross-model comparison pending — requires different generator runs) | — |
| C6 | Scoped models limited by evidence, not scope quality | See Sec 2 — M4 intersection vs M2/M3 comparison | Method comparison table |

C3 and C4 are the main testable claims from the generation-level evaluation. C1 and C2 require retrieval-level or formula-level analysis (separate notebook). C5 requires multiple generator models. C6 is observable from the method ranking.

## 10. Reproducibility

- **Data**: `results/sikdd_replication.csv` — produced by `experiments/parse_to_flat_csv.py`
- **Source JSONs**: `results/{experiment_id}/*_details.json` — produced by `experiments/04_evaluate.py`
- **Feedback DB**: `data/processed/feedback_conditioned.db` (878 train queries, 175,600 rows, gpt-5.6-luna judge)
- **FAISS index**: `data/processed/faiss_index/` (all-MiniLM-L6-v2, 1,595 vectors)
- **Split**: `data/processed/splits/split_seed42.json` (319 dev queries)
- **Generator**: gpt-5.6-luna, temperature 0
- **Figures saved to**: `notebooks/figures/`